##  Project Documentation -> Please open the document for full clarity. There are three tabs on the left side of the Google Docs.

🔗 https://docs.google.com/document/d/1OocowrXOJG_5V0GSSvzRcpaHOyxwygmT2YSZm5gSVIQ/edit?usp=sharing

The 4-page **summary document** is provided in the link above. In the Google Docs file, you will see **three tabs on the left-hand side**:

1. **Summary Document** – A concise 4-page overview of the entire project and the key results.
2. **ViT Pipeline** – A detailed explanation of the **Vision Transformer–based pipeline** implemented for person re-identification.
3. **ResNet Pipeline** – A detailed explanation of the **ResNet-based pipeline** and the training strategies used.

The **summary document** also contains links to other important **Kaggle notebooks** related to this project.

[Open the full document](https://docs.google.com/document/d/1OocowrXOJG_5V0GSSvzRcpaHOyxwygmT2YSZm5gSVIQ/edit?usp=sharing)

# Training Pipeline

This notebook contains the **training pipeline of a ResNet50 backbone** for Person Re-Identification.

It achieves **Best Rank-1: 94.30%** and **Best mAP: 85.70%** on the **Market-1501 dataset** when trained for **120 epochs**.

The **complete configuration** used for training is provided **clearly in the first cell of the notebook**.

# Model Checkpoint and Inference

I have also provided the **model checkpoint** containing the **model weights at epoch 120**, which corresponds to the **best-performing model**.

Using this checkpoint, we run **inference** and then compute:

- **VRAM usage**
- **Model computation cost**

Additionally, we will also perform **model quantization** for further analysis.

In [ ]:
# =============================================================================
# Market-1501 Dataset
# =============================================================================



# ── Dataset ──────────────────────────────────────────────────────────────────
DATASET_ROOT        = '/kaggle/input/datasets/rayiooo/reid_market-1501'
DATASET_PERCENT     = 1.0     # 0.05 = use only 5% of data (for debug runs)
                               # 1.0  = full dataset

# ── Image Input ──────────────────────────────────────────────────────────────
IMAGE_HEIGHT        = 256
IMAGE_WIDTH         = 128
RANDOM_FLIP_PROB    = 0.5     # horizontal flip probability
PADDING             = 10      # zero-pad before random crop
PIXEL_MEAN          = [0.485, 0.456, 0.406]
PIXEL_STD           = [0.229, 0.224, 0.225]

# ── Random Erasing Augmentation  ────────────────────────────────────
RE_PROB             = 0.5     # probability of applying random erasing
RE_SL               = 0.02    # min erasing area ratio
RE_SH               = 0.4     # max erasing area ratio
RE_R1               = 0.3     # min aspect ratio of erased region

# ── Batch Sampling (P×K) ─────────────────────────────────────────────────────
P_IDENTITIES        = 16      # number of identities per batch
K_IMAGES            = 4       # number of images per identity
# batch size = P_IDENTITIES * K_IMAGES = 64
TEST_BATCH_SIZE     = 128

# ── Model ─────────────────────────────────────────────────────────────────────
LAST_STRIDE         = 1       
NECK                = 'bnneck'  # 
NECK_FEAT           = 'after'   # 'after'=use fi (after BN) at inference
FEAT_NORM           = True    # L2-normalize features at test time

# ── Loss Weights ─────────────────────────────────────────────────────────────
LABEL_SMOOTH        = True    # 
LABEL_SMOOTH_EPS    = 0.1     # ε for label smoothing
TRIPLET_MARGIN      = 0.3     #  (baseline): triplet loss margin
USE_CENTER_LOSS     = True    # : center loss
CENTER_LOSS_WEIGHT  = 0.0005  # β in paper (L = L_id + L_tri + β*L_center)
CENTER_LR           = 0.5     # SGD lr for center loss parameters

# ── Optimizer & Scheduler ────────────────────────────────────────────────────
BASE_LR             = 0.00035
WEIGHT_DECAY        = 0.0005
BIAS_LR_FACTOR      = 1.0
WEIGHT_DECAY_BIAS   = 0.0005
OPTIMIZER           = 'Adam'  # 'Adam' or 'SGD'

# ── Warmup LR Schedule () ─────────────────────────────────────────────
MAX_EPOCHS          = 120
WARMUP_EPOCHS       = 10      # linearly ramp lr from WARMUP_START_LR to BASE_LR
WARMUP_START_LR     = 3.5e-5  # starting lr during warmup
MILESTONES          = [40, 70] # epochs at which lr is multiplied by GAMMA
GAMMA               = 0.1     # lr decay factor at each milestone

# ── Checkpoint & Resume ──────────────────────────────────────────────────────
OUTPUT_DIR          = '/kaggle/working/reid_output'
CHECKPOINT_FREQ     = 10      # save checkpoint every N epochs
RESUME_CHECKPOINT   = ''      # path to checkpoint to resume from, '' = start fresh
                               # e.g. '/kaggle/working/reid_output/checkpoint_epoch40.pth'

# ── Evaluation ───────────────────────────────────────────────────────────────
EVAL_FREQ           = 10      # run validation every N epochs
MAX_RANK            = 50      # max CMC rank to compute

# ── Misc ──────────────────────────────────────────────────────────────────────
NUM_WORKERS         = 4
SEED                = 42
DEVICE              = 'cuda'  # 'cuda' or 'cpu'

# =============================================================================
# IMPORTS
# =============================================================================

import os
import re
import copy
import math
import random
import json
import time
import glob
from collections import defaultdict
from bisect import bisect_right

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
import torchvision.transforms as T
from torchvision.models import resnet50, ResNet50_Weights
from PIL import Image
import scipy.io
from tqdm import tqdm


In [ ]:
# =============================================================================
# REPRODUCIBILITY
# =============================================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)
DEVICE = torch.device(DEVICE if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# =============================================================================
# DATASET — Market-1501 Parser
# =============================================================================

def parse_market1501_image(img_path):
    """
    Market-1501 filename format: PPPP_cCsSS_FFFFFF_II.jpg
    PPPP = person id  (e.g. 0002),  -1 = junk/distractor
    cC   = camera id  (e.g. c1)
    """
    fname = os.path.basename(img_path)
    pid_str = fname.split('_')[0]
    cam_str = fname.split('_')[1]  # e.g. 'c1s1'
    pid = int(pid_str)
    camid = int(cam_str[1]) - 1   # 0-indexed
    return pid, camid


class Market1501(object):
    """
    Parse Market-1501 directory structure into:
        train  : list of (img_path, pid, camid)
        query  : list of (img_path, pid, camid)
        gallery: list of (img_path, pid, camid)
    """
    def __init__(self, root, data_percent=1.0):
        self.root       = root
        self.data_pct   = data_percent

        self.train   = self._load_split('bounding_box_train', is_train=True)
        self.query   = self._load_split('query',              is_train=False)
        self.gallery = self._load_split('bounding_box_test',  is_train=False)

        self.num_train_pids = self._count_pids(self.train)
        self.num_query_pids = self._count_pids(self.query)
        self.num_gallery_pids = self._count_pids(self.gallery)

        print(f"\nDataset loaded  [using {data_percent*100:.1f}% of train]")
        print(f"  Train  : {len(self.train):5d} images | {self.num_train_pids} IDs")
        print(f"  Query  : {len(self.query):5d} images | {self.num_query_pids} IDs")
        print(f"  Gallery: {len(self.gallery):5d} images | {self.num_gallery_pids} IDs\n")

    def _load_split(self, folder, is_train):
        img_dir = os.path.join(self.root, folder)
        paths   = sorted(glob.glob(os.path.join(img_dir, '*.jpg')))
        data    = []

        pid_set = {}  # maps original pid → re-labelled pid (train only)

        for p in paths:
            pid, camid = parse_market1501_image(p)
            if pid == -1:       # skip junk/distractors
                continue
            if is_train:
                if pid not in pid_set:
                    pid_set[pid] = len(pid_set)
                pid = pid_set[pid]  # re-label 0..N-1
            data.append((p, pid, camid))

        # Apply dataset percentage (train split only, stratified by identity)
        if is_train and self.data_pct < 1.0:
            data = self._subsample(data, self.data_pct)
            # Re-relabel pids after subsampling
            pids = sorted(set(d[1] for d in data))
            pid_remap = {p: i for i, p in enumerate(pids)}
            data = [(p, pid_remap[pid], cam) for p, pid, cam in data]

        return data

    def _subsample(self, data, pct):
        """Keep `pct` fraction of identities (all images of chosen IDs)."""
        by_pid = defaultdict(list)
        for item in data:
            by_pid[item[1]].append(item)
        pids = sorted(by_pid.keys())
        n_keep = max(1, int(len(pids) * pct))
        chosen = random.sample(pids, n_keep)
        result = []
        for pid in chosen:
            result.extend(by_pid[pid])
        return result

    def _count_pids(self, data):
        return len(set(d[1] for d in data))



# Random Identity Sampler (P×K)

In [ ]:
# =============================================================================
# DATASET — PyTorch Dataset Wrapper
# =============================================================================

class ImageDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data      = data       # list of (path, pid, camid)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, pid, camid = self.data[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, pid, camid, path


# =============================================================================
# TRANSFORMS — including Random Erasing 
# =============================================================================

class RandomErasing:
    """
    Random Erasing Data Augmentation — Zhong et al. 2017
    https://arxiv.org/pdf/1708.04896.pdf
    """
    def __init__(self, probability=RE_PROB, sl=RE_SL, sh=RE_SH,
                 r1=RE_R1, mean=PIXEL_MEAN):
        self.prob  = probability
        self.sl    = sl
        self.sh    = sh
        self.r1    = r1
        self.mean  = mean

    def __call__(self, img):
        if random.uniform(0, 1) >= self.prob:
            return img
        for _ in range(100):
            area = img.size(1) * img.size(2)
            target_area  = random.uniform(self.sl, self.sh) * area
            aspect_ratio = random.uniform(self.r1, 1 / self.r1)
            h = int(round(math.sqrt(target_area * aspect_ratio)))
            w = int(round(math.sqrt(target_area / aspect_ratio)))
            if w < img.size(2) and h < img.size(1):
                x1 = random.randint(0, img.size(1) - h)
                y1 = random.randint(0, img.size(2) - w)
                img[0, x1:x1+h, y1:y1+w] = self.mean[0]
                img[1, x1:x1+h, y1:y1+w] = self.mean[1]
                img[2, x1:x1+h, y1:y1+w] = self.mean[2]
                return img
        return img


def build_transforms(is_train=True):
    normalize = T.Normalize(mean=PIXEL_MEAN, std=PIXEL_STD)
    if is_train:
        return T.Compose([
            T.Resize((IMAGE_HEIGHT, IMAGE_WIDTH)),
            T.RandomHorizontalFlip(p=RANDOM_FLIP_PROB),
            T.Pad(PADDING),
            T.RandomCrop((IMAGE_HEIGHT, IMAGE_WIDTH)),
            T.ToTensor(),
            normalize,
            RandomErasing()        
        ])
    else:
        return T.Compose([
            T.Resize((IMAGE_HEIGHT, IMAGE_WIDTH)),
            T.ToTensor(),
            normalize
        ])


# =============================================================================
# SAMPLER — Random Identity Sampler (P×K)
# =============================================================================

class RandomIdentitySampler(Sampler):
    """
    Sample P identities, K images each → batch = P*K images.
    Used for triplet loss training.
    """
    def __init__(self, data, batch_size, num_instances):
        self.data          = data
        self.batch_size    = batch_size
        self.num_instances = num_instances
        self.num_pids_per_batch = batch_size // num_instances

        self.index_by_pid = defaultdict(list)
        for idx, (_, pid, _) in enumerate(data):
            self.index_by_pid[pid].append(idx)
        self.pids = list(self.index_by_pid.keys())

    def __iter__(self):
        # For every pid, chunk its indices into groups of K
        batch_pool = defaultdict(list)
        for pid in self.pids:
            idxs = copy.deepcopy(self.index_by_pid[pid])
            if len(idxs) < self.num_instances:
                idxs = list(np.random.choice(idxs, self.num_instances, replace=True))
            random.shuffle(idxs)
            chunk = []
            for idx in idxs:
                chunk.append(idx)
                if len(chunk) == self.num_instances:
                    batch_pool[pid].append(chunk)
                    chunk = []

        avail_pids = copy.deepcopy(self.pids)
        final_idxs = []

        while len(avail_pids) >= self.num_pids_per_batch:
            selected = random.sample(avail_pids, self.num_pids_per_batch)
            for pid in selected:
                chunk = batch_pool[pid].pop(0)
                final_idxs.extend(chunk)
                if len(batch_pool[pid]) == 0:
                    avail_pids.remove(pid)

        return iter(final_idxs)

    def __len__(self):
        total = 0
        for pid in self.pids:
            n = len(self.index_by_pid[pid])
            if n < self.num_instances:
                n = self.num_instances
            total += n - (n % self.num_instances)
        return total


# =============================================================================
# COLLATE FUNCTIONS
# =============================================================================

def train_collate(batch):
    imgs, pids, camids, _ = zip(*batch)
    return torch.stack(imgs), torch.tensor(pids, dtype=torch.long)


def val_collate(batch):
    imgs, pids, camids, paths = zip(*batch)
    return torch.stack(imgs), list(pids), list(camids)



# ResNet50 with BNNeck 

In [ ]:
# =============================================================================
# DATA LOADERS
# =============================================================================

def build_dataloaders(dataset):
    batch_size  = P_IDENTITIES * K_IMAGES

    train_set   = ImageDataset(dataset.train, build_transforms(is_train=True))
    val_set     = ImageDataset(dataset.query + dataset.gallery,
                               build_transforms(is_train=False))

    train_loader = DataLoader(
        train_set,
        batch_size=batch_size,
        sampler=RandomIdentitySampler(dataset.train, batch_size, K_IMAGES),
        num_workers=NUM_WORKERS,
        collate_fn=train_collate,
        pin_memory=True,
        drop_last=True
    )

    val_loader = DataLoader(
        val_set,
        batch_size=TEST_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=val_collate,
        pin_memory=True
    )

    return train_loader, val_loader


# =============================================================================
# MODEL — ResNet50 with BNNeck 
# =============================================================================

def weights_init_kaiming(m):
    cls = m.__class__.__name__
    if 'Linear' in cls:
        nn.init.kaiming_normal_(m.weight, a=0, mode='fan_out')
        nn.init.constant_(m.bias, 0.0)
    elif 'Conv' in cls:
        nn.init.kaiming_normal_(m.weight, a=0, mode='fan_in')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0.0)
    elif 'BatchNorm' in cls:
        if m.affine:
            nn.init.constant_(m.weight, 1.0)
            nn.init.constant_(m.bias, 0.0)


def weights_init_classifier(m):
    if 'Linear' in m.__class__.__name__:
        nn.init.normal_(m.weight, std=0.001)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0.0)


class ReIDModel(nn.Module):
    """
    ResNet50 backbone
    + Global Average Pooling
    + BNNeck 
    + ID classifier (no bias, Kaiming init)
    """
    in_planes = 2048

    def __init__(self, num_classes, last_stride=LAST_STRIDE,
                 neck=NECK, neck_feat=NECK_FEAT):
        super().__init__()
        self.neck_type = neck
        self.neck_feat = neck_feat

        # ── Backbone ──────────────────────────────────────────────────────────
        base = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

        # : change last stride from 2 → 1
        base.layer4[0].conv2.stride      = (last_stride, last_stride)
        base.layer4[0].downsample[0].stride = (last_stride, last_stride)

        # Remove the final fc & avgpool from torchvision resnet
        self.backbone = nn.Sequential(*list(base.children())[:-2])

        # ── Pooling ───────────────────────────────────────────────────────────
        self.gap = nn.AdaptiveAvgPool2d(1)

        # ── BNNeck ──────────────────────────────────────────────────
        if self.neck_type == 'bnneck':
            self.bottleneck = nn.BatchNorm1d(self.in_planes)
            self.bottleneck.bias.requires_grad_(False)   # no bias shift
            self.bottleneck.apply(weights_init_kaiming)

        # ── Classifier ────────────────────────────────────────────────────────
        self.classifier = nn.Linear(self.in_planes, num_classes, bias=False)
        self.classifier.apply(weights_init_classifier)

    def forward(self, x):
        # Feature map from backbone
        x      = self.backbone(x)          # (B, 2048, H, W)
        ft     = self.gap(x)               # (B, 2048, 1, 1)
        ft     = ft.view(ft.shape[0], -1)  # (B, 2048) — feature before BN

        if self.neck_type == 'bnneck':
            fi = self.bottleneck(ft)       # (B, 2048) — feature after BN
        else:
            fi = ft

        if self.training:
            cls_score = self.classifier(fi)
            return cls_score, ft           # ft for triplet, fi for ID loss
        else:
            # Inference: return fi (after BN) or ft (before BN)
            if self.neck_feat == 'after':
                return fi
            return ft



 # LR SCHEDULER — Warmup + MultiStep

In [ ]:
# =============================================================================
# LOSSES
# =============================================================================

# ── Label Smoothing Cross Entropy ───────────────────────────────────

class CrossEntropyLabelSmooth(nn.Module):
    """
    Cross entropy with label smoothing.
    y_smooth = (1-ε)*y_one_hot + ε/N
    """
    def __init__(self, num_classes, eps=LABEL_SMOOTH_EPS):
        super().__init__()
        self.num_classes = num_classes
        self.eps         = eps
        self.logsoftmax  = nn.LogSoftmax(dim=1)

    def forward(self, inputs, targets):
        log_probs = self.logsoftmax(inputs)
        # One-hot encode targets
        targets_oh = torch.zeros_like(log_probs).scatter_(
            1, targets.unsqueeze(1), 1)
        # Smooth
        targets_oh = (1 - self.eps) * targets_oh + \
                     self.eps / self.num_classes
        loss = (-targets_oh * log_probs).mean(0).sum()
        return loss


# ── Triplet Loss with Hard Mining ────────────────────────────────────────────

def euclidean_dist(x, y):
    """Pairwise euclidean distance. x: (m,d), y: (n,d) → (m,n)"""
    m, n = x.size(0), y.size(0)
    xx = torch.pow(x, 2).sum(1, keepdim=True).expand(m, n)
    yy = torch.pow(y, 2).sum(1, keepdim=True).expand(n, m).t()
    dist = (xx + yy).addmm_(x, y.t(), beta=1, alpha=-2).clamp(min=1e-12).sqrt()
    return dist


def hard_example_mining(dist_mat, labels):
    """
    For each anchor find:
      - hardest positive  (same ID, max distance)
      - hardest negative  (different ID, min distance)
    """
    N   = dist_mat.size(0)
    is_pos = labels.expand(N, N).eq(labels.expand(N, N).t())
    is_neg = labels.expand(N, N).ne(labels.expand(N, N).t())

    dist_ap = dist_mat[is_pos].contiguous().view(N, -1).max(1)[0]
    dist_an = dist_mat[is_neg].contiguous().view(N, -1).min(1)[0]
    return dist_ap, dist_an


class TripletLoss(nn.Module):
    """
    Triplet loss with hard mining.
    margin > 0 → MarginRankingLoss
    """
    def __init__(self, margin=TRIPLET_MARGIN):
        super().__init__()
        self.margin       = margin
        self.ranking_loss = nn.MarginRankingLoss(margin=margin)

    def forward(self, feats, labels):
        dist_mat        = euclidean_dist(feats, feats)
        dist_ap, dist_an = hard_example_mining(dist_mat, labels)
        y = torch.ones_like(dist_an)
        loss = self.ranking_loss(dist_an, dist_ap, y)
        return loss, dist_ap, dist_an


# ── : Center Loss ──────────────────────────────────────────────────────

class CenterLoss(nn.Module):
    """
    Center loss — Wen et al. ECCV 2016.
    Penalises distance of features from their class centres.
    Centres are learned alongside the model.
    """
    def __init__(self, num_classes, feat_dim=2048):
        super().__init__()
        self.num_classes = num_classes
        self.feat_dim    = feat_dim
        self.centers     = nn.Parameter(
            torch.randn(num_classes, feat_dim).to(DEVICE))

    def forward(self, feats, labels):
        bs  = feats.size(0)
        # (bs, num_classes)
        distmat = (torch.pow(feats, 2).sum(1, keepdim=True).expand(bs, self.num_classes)
                   + torch.pow(self.centers, 2).sum(1, keepdim=True).expand(
                       self.num_classes, bs).t())
        distmat.addmm_(feats, self.centers.t(), beta=1, alpha=-2)

        classes = torch.arange(self.num_classes).long().to(DEVICE)
        mask    = labels.unsqueeze(1).expand(bs, self.num_classes).eq(
                  classes.expand(bs, self.num_classes))

        dist = distmat * mask.float()
        loss = dist.clamp(min=1e-12, max=1e+12).sum() / bs
        return loss


# =============================================================================
# COMBINED LOSS FUNCTION
# =============================================================================

class ReIDLoss(nn.Module):
    """
    L = L_id  +  L_triplet  +  β * L_center
    """
    def __init__(self, num_classes):
        super().__init__()
        self.id_loss      = CrossEntropyLabelSmooth(num_classes) \
                            if LABEL_SMOOTH else nn.CrossEntropyLoss()
        self.triplet_loss = TripletLoss(margin=TRIPLET_MARGIN)
        if USE_CENTER_LOSS:
            self.center_loss  = CenterLoss(num_classes)
        self.use_center   = USE_CENTER_LOSS
        self.beta         = CENTER_LOSS_WEIGHT

    def forward(self, cls_score, feat, target):
        l_id  = self.id_loss(cls_score, target)
        l_tri, dist_ap, dist_an = self.triplet_loss(feat, target)
        loss  = l_id + l_tri
        if self.use_center:
            l_cen  = self.center_loss(feat, target)
            loss  += self.beta * l_cen
        else:
            l_cen = torch.tensor(0.0)
        return loss, l_id, l_tri, l_cen


# =============================================================================
# LR SCHEDULER — Warmup + MultiStep 
# =============================================================================

class WarmupMultiStepLR(torch.optim.lr_scheduler._LRScheduler):
    """
    Linear warmup for `warmup_epochs`, then MultiStep decay.
    epoch 0..warmup_epochs : lr linearly from warmup_start_lr → base_lr
    epoch milestones       : lr *= gamma
    """
    def __init__(self, optimizer, milestones, gamma=GAMMA,
                 warmup_epochs=WARMUP_EPOCHS,
                 warmup_start_lr=WARMUP_START_LR,
                 last_epoch=-1):
        self.milestones      = milestones
        self.gamma           = gamma
        self.warmup_epochs   = warmup_epochs
        self.warmup_start_lr = warmup_start_lr
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        ep = self.last_epoch
        if ep < self.warmup_epochs:
            # linear ramp
            alpha = ep / self.warmup_epochs
            factor = self.warmup_start_lr / self.base_lrs[0] * (1 - alpha) + alpha
        else:
            factor = self.gamma ** bisect_right(self.milestones, ep)
        return [base_lr * factor for base_lr in self.base_lrs]


# =============================================================================
# OPTIMIZER BUILDER
# =============================================================================

def build_optimizer(model):
    params = []
    for key, val in model.named_parameters():
        if not val.requires_grad:
            continue
        lr = BASE_LR
        wd = WEIGHT_DECAY
        if 'bias' in key:
            lr = BASE_LR * BIAS_LR_FACTOR
            wd = WEIGHT_DECAY_BIAS
        params.append({'params': [val], 'lr': lr, 'weight_decay': wd})
    if OPTIMIZER == 'SGD':
        return torch.optim.SGD(params, momentum=0.9)
    return torch.optim.Adam(params)


# =============================================================================
# EVALUATION — CMC & mAP
# =============================================================================

def eval_market1501(distmat, q_pids, g_pids, q_camids, g_camids,
                    max_rank=MAX_RANK):
    """
    Standard Market-1501 evaluation protocol.
    Returns:
        cmc  : np.ndarray, shape (max_rank,)
        mAP  : float
    """
    num_q, num_g = distmat.shape
    if num_g < max_rank:
        max_rank = num_g

    indices   = np.argsort(distmat, axis=1)
    all_cmc   = []
    all_AP    = []

    for q_idx in range(num_q):
        q_pid   = q_pids[q_idx]
        q_camid = q_camids[q_idx]

        order   = indices[q_idx]
        # Remove same-camera same-id (junk)
        remove  = (g_pids[order] == q_pid) & (g_camids[order] == q_camid)
        keep    = np.invert(remove)

        raw_cmc = (g_pids[order][keep] == q_pid).astype(np.int32)

        if raw_cmc.sum() == 0:
            # Query not in gallery — skip
            continue

        # CMC
        cmc = raw_cmc.cumsum()
        cmc[cmc > 1] = 1
        all_cmc.append(cmc[:max_rank])

        # Average Precision
        num_rel    = raw_cmc.sum()
        tmp_cmc    = raw_cmc.cumsum()
        tmp_cmc    = [x / (i + 1.) for i, x in enumerate(tmp_cmc)]
        tmp_cmc    = np.asarray(tmp_cmc) * raw_cmc
        AP         = tmp_cmc.sum() / num_rel
        all_AP.append(AP)

    all_cmc = np.asarray(all_cmc).astype(np.float32)
    cmc     = all_cmc.mean(axis=0)
    mAP     = np.mean(all_AP)
    return cmc, mAP


@torch.no_grad()
def run_evaluation(model, val_loader, num_query):
    model.eval()
    feats_list, pids_list, camids_list = [], [], []

    for imgs, pids, camids in tqdm(val_loader, desc='Extracting features'):
        imgs = imgs.to(DEVICE)
        feat = model(imgs)
        if FEAT_NORM:
            feat = F.normalize(feat, dim=1, p=2)
        feats_list.append(feat.cpu())
        pids_list.extend(pids)
        camids_list.extend(camids)

    feats   = torch.cat(feats_list, dim=0)
    q_feats = feats[:num_query]
    g_feats = feats[num_query:]
    q_pids  = np.array(pids_list[:num_query])
    g_pids  = np.array(pids_list[num_query:])
    q_cams  = np.array(camids_list[:num_query])
    g_cams  = np.array(camids_list[num_query:])

    m, n = q_feats.shape[0], g_feats.shape[0]
    distmat = (torch.pow(q_feats, 2).sum(1, keepdim=True).expand(m, n)
               + torch.pow(g_feats, 2).sum(1, keepdim=True).expand(n, m).t())
    distmat.addmm_(q_feats, g_feats.t(), beta=1, alpha=-2)
    distmat = distmat.numpy()

    cmc, mAP = eval_market1501(distmat, q_pids, g_pids, q_cams, g_cams)
    return cmc, mAP


# TRAINING LOOP

In [ ]:

# =============================================================================
# CHECKPOINT UTILITIES
# =============================================================================

def save_checkpoint(epoch, model, optimizer, optimizer_center,
                    history, path):
    """Save everything needed to resume training."""
    state = {
        'epoch'             : epoch,
        'model_state'       : model.state_dict(),
        'optimizer_state'   : optimizer.state_dict(),
        'history'           : history,
    }
    if optimizer_center is not None:
        state['optimizer_center_state'] = optimizer_center.state_dict()
    torch.save(state, path)
    print(f"  ✓ Checkpoint saved → {path}")


def load_checkpoint(path, model, optimizer, optimizer_center):
    """Load checkpoint and return (start_epoch, history)."""
    print(f"Resuming from checkpoint: {path}")
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    if optimizer_center is not None and 'optimizer_center_state' in ckpt:
        optimizer_center.load_state_dict(ckpt['optimizer_center_state'])
    history     = ckpt.get('history', init_history())
    start_epoch = ckpt['epoch'] + 1
    print(f"  Resumed at epoch {start_epoch}")
    return start_epoch, history


def init_history():
    return {
        'train_loss'     : [],   # total loss per epoch
        'train_id_loss'  : [],
        'train_tri_loss' : [],
        'train_cen_loss' : [],
        'train_acc'      : [],   # ID classification accuracy per epoch
        'val_rank1'      : [],   # Rank-1 accuracy (at eval epochs)
        'val_rank5'      : [],
        'val_rank10'     : [],
        'val_mAP'        : [],
        'eval_epochs'    : [],   # which epochs had evaluation
        'epoch_times'    : [],   # seconds per epoch
    }


def save_history_json(history):
    path = os.path.join(OUTPUT_DIR, 'training_history.json')
    with open(path, 'w') as f:
        json.dump(history, f, indent=2)


# =============================================================================
# TRAINING LOOP
# =============================================================================

def train_one_epoch(model, loss_fn, optimizer, optimizer_center,
                    train_loader, epoch):
    model.train()
    total_loss = total_id = total_tri = total_cen = total_acc = 0.0
    n_batches  = 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch:3d}/{MAX_EPOCHS}',
                leave=False)
    for imgs, pids in pbar:
        imgs = imgs.to(DEVICE)
        pids = pids.to(DEVICE)

        optimizer.zero_grad()
        if USE_CENTER_LOSS:
            optimizer_center.zero_grad()

        cls_score, feat = model(imgs)
        loss, l_id, l_tri, l_cen = loss_fn(cls_score, feat, pids)

        loss.backward()
        optimizer.step()

        # Center loss: scale gradients before optimizer step
        if USE_CENTER_LOSS:
            for param in loss_fn.center_loss.parameters():
                param.grad.data *= (1. / CENTER_LOSS_WEIGHT)
            optimizer_center.step()

        # Accuracy (ID classification)
        acc = (cls_score.max(1)[1] == pids).float().mean().item()

        total_loss += loss.item()
        total_id   += l_id.item()
        total_tri  += l_tri.item()
        total_cen  += l_cen.item() if USE_CENTER_LOSS else 0.0
        total_acc  += acc
        n_batches  += 1

        pbar.set_postfix({
            'loss': f'{loss.item():.3f}',
            'acc' : f'{acc:.3f}',
            'lr'  : f'{optimizer.param_groups[0]["lr"]:.2e}'
        })

    return {
        'loss'    : total_loss / n_batches,
        'id_loss' : total_id   / n_batches,
        'tri_loss': total_tri  / n_batches,
        'cen_loss': total_cen  / n_batches,
        'acc'     : total_acc  / n_batches,
    }


def train(dataset):
    # ── Build loaders ─────────────────────────────────────────────────────────
    train_loader, val_loader = build_dataloaders(dataset)
    num_query = len(dataset.query)

    # ── Build model ───────────────────────────────────────────────────────────
    model = ReIDModel(num_classes=dataset.num_train_pids).to(DEVICE)

    # ── Loss ──────────────────────────────────────────────────────────────────
    loss_fn = ReIDLoss(dataset.num_train_pids).to(DEVICE)

    # ── Optimizers ────────────────────────────────────────────────────────────
    optimizer        = build_optimizer(model)
    optimizer_center = None
    if USE_CENTER_LOSS:
        optimizer_center = torch.optim.SGD(
            loss_fn.center_loss.parameters(), lr=CENTER_LR)

    # ── Scheduler ─────────────────────────────────────────────────────────────
    scheduler = WarmupMultiStepLR(optimizer, MILESTONES)

    # ── Resume or fresh start ─────────────────────────────────────────────────
    history     = init_history()
    start_epoch = 1
    if RESUME_CHECKPOINT and os.path.isfile(RESUME_CHECKPOINT):
        start_epoch, history = load_checkpoint(
            RESUME_CHECKPOINT, model, optimizer, optimizer_center)
        # Fast-forward scheduler
        for _ in range(start_epoch - 1):
            scheduler.step()

    print(f"\nStarting training from epoch {start_epoch}")
    print(f"  Identities : {dataset.num_train_pids}")
    print(f"  Train imgs : {len(dataset.train)}")
    print(f"  Batch size : {P_IDENTITIES * K_IMAGES}  (P={P_IDENTITIES}, K={K_IMAGES})")
    print(f"  Tricks     : warmup={WARMUP_EPOCHS}ep, REA, "
          f"label_smooth={LABEL_SMOOTH}, stride={LAST_STRIDE}, "
          f"BNNeck={NECK}, center={USE_CENTER_LOSS}\n")

    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        t0 = time.time()

        # ── Step scheduler at epoch START (matches original code behaviour) ───
        scheduler.step()

        # ── Train ─────────────────────────────────────────────────────────────
        stats = train_one_epoch(model, loss_fn, optimizer,
                                optimizer_center, train_loader, epoch)

        elapsed = time.time() - t0

        # ── Log ───────────────────────────────────────────────────────────────
        history['train_loss'].append(stats['loss'])
        history['train_id_loss'].append(stats['id_loss'])
        history['train_tri_loss'].append(stats['tri_loss'])
        history['train_cen_loss'].append(stats['cen_loss'])
        history['train_acc'].append(stats['acc'])
        history['epoch_times'].append(elapsed)

        print(f"Epoch {epoch:3d}/{MAX_EPOCHS} | "
              f"Loss: {stats['loss']:.4f} "
              f"(id={stats['id_loss']:.3f}, "
              f"tri={stats['tri_loss']:.3f}, "
              f"cen={stats['cen_loss']:.4f}) | "
              f"Acc: {stats['acc']:.3f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.2e} | "
              f"Time: {elapsed:.1f}s")

        # ── Evaluation ────────────────────────────────────────────────────────
        if epoch % EVAL_FREQ == 0 or epoch == MAX_EPOCHS:
            cmc, mAP = run_evaluation(model, val_loader, num_query)
            history['val_rank1'].append(float(cmc[0]))
            history['val_rank5'].append(float(cmc[4]))
            history['val_rank10'].append(float(cmc[9]))
            history['val_mAP'].append(float(mAP))
            history['eval_epochs'].append(epoch)
            print(f"  ── Val @ epoch {epoch} ──")
            print(f"     mAP   : {mAP:.4f}  ({mAP*100:.2f}%)")
            print(f"     Rank-1: {cmc[0]:.4f}  ({cmc[0]*100:.2f}%)")
            print(f"     Rank-5: {cmc[4]:.4f}  ({cmc[4]*100:.2f}%)")
            print(f"     Rank-10:{cmc[9]:.4f}  ({cmc[9]*100:.2f}%)")

        # ── Checkpoint ────────────────────────────────────────────────────────
        if epoch % CHECKPOINT_FREQ == 0 or epoch == MAX_EPOCHS:
            ckpt_path = os.path.join(OUTPUT_DIR, f'checkpoint_epoch{epoch:03d}.pth')
            save_checkpoint(epoch, model, optimizer, optimizer_center,
                            history, ckpt_path)

        # ── Save history JSON after every epoch ───────────────────────────────
        save_history_json(history)

    print("\n✓ Training complete.")
    print(f"  Best Rank-1: {max(history['val_rank1'])*100:.2f}%")
    print(f"  Best mAP   : {max(history['val_mAP'])*100:.2f}%")
    return model, history

# PLOTS

In [ ]:
# =============================================================================
# PLOTS
# =============================================================================

def plot_training_history(history):
    """
    Two figures saved to OUTPUT_DIR and displayed inline:
      Figure 1 — Training curves  (loss breakdown + accuracy)
      Figure 2 — Validation curves (Rank-1, Rank-5, Rank-10, mAP)
    """
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker

    train_epochs = list(range(1, len(history['train_loss']) + 1))
    eval_epochs  = history['eval_epochs']

    # ── Figure 1 : Training ───────────────────────────────────────────────────
    fig1, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig1.suptitle('Training Curves', fontsize=14, fontweight='bold')

    # Left — Loss breakdown
    ax = axes[0]
    ax.plot(train_epochs, history['train_loss'],     label='Total Loss',    linewidth=2)
    ax.plot(train_epochs, history['train_id_loss'],  label='ID Loss',       linestyle='--')
    ax.plot(train_epochs, history['train_tri_loss'], label='Triplet Loss',  linestyle='--')
    if USE_CENTER_LOSS:
        ax.plot(train_epochs, history['train_cen_loss'],
                label=f'Center Loss (×{CENTER_LOSS_WEIGHT})', linestyle=':')
    for ms in MILESTONES:
        ax.axvline(ms, color='grey', linestyle=':', alpha=0.6,
                   label=f'LR decay @ ep{ms}' if ms == MILESTONES[0] else '')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Loss per Epoch')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Right — Training accuracy
    ax = axes[1]
    ax.plot(train_epochs, [a * 100 for a in history['train_acc']],
            color='darkorange', linewidth=2, label='Train ID Acc')
    for ms in MILESTONES:
        ax.axvline(ms, color='grey', linestyle=':', alpha=0.6)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('ID Classification Accuracy')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    fig1.tight_layout()
    p1 = os.path.join(OUTPUT_DIR, 'plot_training.png')
    fig1.savefig(p1, dpi=120, bbox_inches='tight')
    print(f"  ✓ Saved → {p1}")
    plt.show()

    # ── Figure 2 : Validation ─────────────────────────────────────────────────
    if not eval_epochs:
        print("  No validation data to plot yet.")
        return

    fig2, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig2.suptitle('Validation Curves (evaluated every '
                  f'{EVAL_FREQ} epochs)', fontsize=14, fontweight='bold')

    # Left — CMC Ranks
    ax = axes[0]
    ax.plot(eval_epochs, [v * 100 for v in history['val_rank1']],
            marker='o', label='Rank-1', linewidth=2)
    ax.plot(eval_epochs, [v * 100 for v in history['val_rank5']],
            marker='s', label='Rank-5', linewidth=2)
    ax.plot(eval_epochs, [v * 100 for v in history['val_rank10']],
            marker='^', label='Rank-10', linewidth=2)
    for ms in MILESTONES:
        ax.axvline(ms, color='grey', linestyle=':', alpha=0.6)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('CMC Accuracy (%)')
    ax.set_title('CMC Rank Accuracy')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # Right — mAP
    ax = axes[1]
    ax.plot(eval_epochs, [v * 100 for v in history['val_mAP']],
            marker='D', color='crimson', linewidth=2, label='mAP')
    for ms in MILESTONES:
        ax.axvline(ms, color='grey', linestyle=':', alpha=0.6)
    # Annotate best mAP
    best_map  = max(history['val_mAP'])
    best_ep   = eval_epochs[history['val_mAP'].index(best_map)]
    ax.annotate(f'Best {best_map*100:.2f}%',
                xy=(best_ep, best_map * 100),
                xytext=(best_ep + 2, best_map * 100 - 3),
                arrowprops=dict(arrowstyle='->', color='black'),
                fontsize=9)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('mAP (%)')
    ax.set_title('Mean Average Precision (mAP)')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    fig2.tight_layout()
    p2 = os.path.join(OUTPUT_DIR, 'plot_validation.png')
    fig2.savefig(p2, dpi=120, bbox_inches='tight')
    print(f"  ✓ Saved → {p2}")
    plt.show()


# =============================================================================
# ENTRY POINT
# =============================================================================

if __name__ == '__main__':
    dataset = Market1501(root=DATASET_ROOT, data_percent=DATASET_PERCENT)
    model, history = train(dataset)

    print("\nGenerating training plots...")
    plot_training_history(history)